# INbreast data exploration

Inventory of the raw INbreast release in `data/raw/inbreast/`: files, patients, DICOM
headers, lesion annotations and mass statistics. Findings are summarized in
`docs/dataset.md`. Outputs are stripped before commit (`nbstripout`).

The file parsing below is exploratory. The reusable version lives in
`src/mammo_seg/data/` (see `docs/data-pipeline.md`). Raw filenames write the
mediolateral oblique view as `ML`; the manifest calls it `MLO`.


In [ ]:
import plistlib
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("pyproject.toml not found above " + str(start))


ROOT = find_repo_root(Path.cwd().resolve())
RAW = ROOT / "data" / "raw" / "inbreast" / "INbreast Release 1.0"
assert RAW.is_dir(), f"INbreast not found at {RAW}"
RNG = np.random.default_rng(0)

## 1. Files, patients and views

The DICOM filename is `<file id>_<patient id>_MG_<laterality>_<view>_ANON`
(see `README.txt`). The patient id is what links images of the same patient.


In [ ]:
FILENAME = re.compile(r"(\d+)_([0-9a-f]+)_MG_([LR])_([A-Z]+)_ANON")


def list_images(raw: Path) -> pd.DataFrame:
    rows = []
    for path in sorted((raw / "AllDICOMs").glob("*.dcm")):
        match = FILENAME.fullmatch(path.stem)
        if match is None:
            raise ValueError(f"unexpected DICOM filename: {path.name}")
        file_id, patient, lat, view = match.groups()
        rows.append(
            {
                "file_id": file_id,
                "patient": patient,
                "lat": lat,
                "view": view,
                "path": path,
            }
        )
    return pd.DataFrame(rows)


images = list_images(RAW)
print(len(images), "images,", images.patient.nunique(), "patients")
print(
    "images per patient:",
    images.groupby("patient").size().value_counts().sort_index().to_dict(),
)
print(images.groupby(["lat", "view"]).size())

In [ ]:
csv = pd.read_csv(RAW / "INbreast.csv", sep=";", dtype=str)
csv["file_id"] = csv["File Name"].str.strip()
merged = images.merge(csv, on="file_id", how="outer", indicator=True)
print(merged["_merge"].value_counts().to_dict())
print("BI-RADS:", csv["Bi-Rads"].value_counts().sort_index().to_dict())

## 2. DICOM headers

In [ ]:
def read_header(path: Path) -> dict:
    ds = pydicom.dcmread(path, stop_before_pixels=True)
    return {
        "rows": ds.Rows,
        "cols": ds.Columns,
        "bits_allocated": ds.BitsAllocated,
        "bits_stored": ds.BitsStored,
        "photometric": ds.PhotometricInterpretation,
        "has_pixel_spacing": "PixelSpacing" in ds,
        "modality": ds.Modality,
    }


headers = pd.DataFrame([read_header(p) for p in images.path])
for column in [
    "rows",
    "cols",
    "bits_allocated",
    "bits_stored",
    "photometric",
    "has_pixel_spacing",
    "modality",
]:
    print(column, headers[column].value_counts().to_dict())
images = pd.concat([images, headers[["rows", "cols"]]], axis=1)

## 3. Lesion annotations

The `.xml` files are Apple property lists exported from OsiriX. Each ROI has a
name, a type and a polygon in pixel coordinates `(x, y)` = `(column, row)`.
Single-point ROIs (type 19) are calcifications marked by one point.


In [ ]:
POINT = re.compile(r"-?\d+\.?\d*")


def polygon_area(points: np.ndarray) -> float:
    if len(points) < 3:
        return 0.0
    x, y = points[:, 0], points[:, 1]
    return 0.5 * abs(float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


def parse_rois(raw: Path) -> pd.DataFrame:
    rows = []
    for xml_path in sorted((raw / "AllXML").glob("*.xml")):
        data = plistlib.loads(xml_path.read_bytes())
        (image,) = data["Images"]
        for roi in image["ROIs"]:
            points = np.array(
                [[float(v) for v in POINT.findall(s)] for s in roi["Point_px"]]
            )
            rows.append(
                {
                    "file_id": xml_path.stem,
                    "name": roi["Name"].strip(),
                    "type": roi["Type"],
                    "n_points": len(points),
                    "area_px": polygon_area(points),
                    "points": points,
                }
            )
    return pd.DataFrame(rows)


rois = parse_rois(RAW).merge(
    images[["file_id", "patient", "rows", "cols"]], on="file_id"
)
print(len(rois), "ROIs in", rois.file_id.nunique(), "of", len(images), "images")
rois.groupby(["name", "type"]).agg(
    n=("file_id", "size"),
    images=("file_id", "nunique"),
    median_points=("n_points", "median"),
    median_area_px=("area_px", "median"),
)

## 4. Masses

Masses are the only lesion type annotated as a proper region in enough images
to train a segmentation model on (see `docs/dataset.md`).


In [ ]:
masses = rois[rois.name == "Mass"].copy()
masses["diameter_px"] = np.sqrt(4 * masses.area_px / np.pi)
masses["image_fraction"] = masses.area_px / (masses.rows * masses.cols)
print(
    len(masses),
    "masses,",
    masses.file_id.nunique(),
    "images,",
    masses.patient.nunique(),
    "patients",
)
print("masses per image:", masses.groupby("file_id").size().value_counts().to_dict())
print(masses[["area_px", "diameter_px", "image_fraction"]].describe().round(4))

per_image = masses.groupby("file_id").image_fraction.sum()
print(f"mass pixels over all images: {100 * per_image.sum() / len(images):.3f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(masses.diameter_px, bins=30)
axes[0].set_xlabel("equivalent diameter (px)")
axes[1].hist(100 * masses.image_fraction, bins=30)
axes[1].set_xlabel("% of image area")
plt.tight_layout()
plt.show()

## 5. Positive and negative images, patients

The project targets mass segmentation, so an image is **positive** when it has
at least one mass and **negative** otherwise. Negatives include images with
only calcifications or other lesions.


In [ ]:
meta = csv[["file_id", "Acquisition date", "ACR", "Bi-Rads"]]
labeled = images.merge(meta, on="file_id")
labeled["n_masses"] = (
    labeled.file_id.map(masses.groupby("file_id").size()).fillna(0).astype(int)
)
labeled["positive"] = labeled.n_masses > 0
labeled["annotated"] = labeled.file_id.isin(rois.file_id)
labeled["size"] = labeled.rows.astype(str) + "x" + labeled.cols.astype(str)

print(
    "positive images:", labeled.positive.sum(), "| negative:", (~labeled.positive).sum()
)
negatives = labeled[~labeled.positive]
print("negatives without any annotation (normal):", (~negatives.annotated).sum())
print("negatives with other lesions only:", negatives.annotated.sum())
for column in ["view", "lat", "size"]:
    print(pd.crosstab(labeled[column], labeled.positive), "\n")
print(
    "masses per positive image:",
    labeled[labeled.positive].n_masses.value_counts().sort_index().to_dict(),
)

In [ ]:
by_patient = labeled.groupby("patient")
patients = pd.DataFrame(
    {
        "images": by_patient.size(),
        "positive_images": by_patient.positive.sum(),
        "exams": by_patient["Acquisition date"].nunique(),
    }
)
patients["kind"] = np.select(
    [patients.positive_images == 0, patients.positive_images == patients.images],
    ["no mass image", "all images positive"],
    default="mixed",
)
print(patients.kind.value_counts().to_dict())
print("exams per patient:", patients.exams.value_counts().sort_index().to_dict())
print(pd.crosstab(patients.images, patients.kind))
print(
    "positive images per positive patient:",
    patients[patients.positive_images > 0]
    .positive_images.value_counts()
    .sort_index()
    .to_dict(),
)

In [ ]:
# Is the same mass visible in both views of a breast?
breasts = labeled.groupby(["patient", "lat", "Acquisition date"]).agg(
    images=("positive", "size"), positive=("positive", "sum")
)
print(pd.crosstab(breasts.images, breasts.positive))

In [ ]:
print(
    "BI-RADS of positive images:",
    labeled[labeled.positive]["Bi-Rads"].value_counts().sort_index().to_dict(),
)
print(
    "ACR density of positive images:",
    labeled[labeled.positive].ACR.str.strip().value_counts().sort_index().to_dict(),
)
print(pd.crosstab(labeled["Acquisition date"], labeled["size"]))

### Mass size after downscaling

Equivalent diameter of the masses at several downscale factors, to choose the
offline resolution. Very small masses (under about 16 px) are hard to segment.


In [ ]:
rows_out = []
for factor in [1, 2, 3, 4, 8, 16]:
    diameter = masses.diameter_px / factor
    rows_out.append(
        {
            "downscale": f"/{factor}",
            "min": diameter.min(),
            "median": diameter.median(),
            "under_16px": int((diameter < 16).sum()),
            "under_32px": int((diameter < 32).sum()),
        }
    )
pd.DataFrame(rows_out).round(0)

### Breast bounding box

The breast covers only part of each image. Cropping to its bounding box before
resizing keeps more resolution for the same cost. Estimated with a simple
intensity threshold (background is close to 0). Reading all 410 images takes
a few minutes, so this uses a random sample; set `SAMPLE = None` for all.


In [ ]:
SAMPLE = 60
THRESHOLD = 100


def breast_bbox(path: Path) -> dict:
    pixels = pydicom.dcmread(path).pixel_array
    foreground = pixels > THRESHOLD
    cols = np.where(foreground.mean(axis=0) > 0.01)[0]
    rows = np.where(foreground.mean(axis=1) > 0.01)[0]
    height, width = rows.max() - rows.min() + 1, cols.max() - cols.min() + 1
    return {
        "breast_area": foreground.mean(),
        "bbox_area": height * width / pixels.size,
        "bbox_height": height,
        "bbox_width": width,
    }


subset = images if SAMPLE is None else images.sample(SAMPLE, random_state=0)
bboxes = pd.DataFrame([breast_bbox(p) for p in subset.path])
bboxes.describe().round(3)

## 6. Visual check of polygons

Red outlines should sit on bright lesions. The image is downsampled by 4 for
display only.


In [ ]:
STEP = 4


def show_masses(file_ids: list[str]) -> None:
    fig, axes = plt.subplots(1, len(file_ids), figsize=(4 * len(file_ids), 6))
    for ax, file_id in zip(np.atleast_1d(axes), file_ids, strict=True):
        path = images.loc[images.file_id == file_id, "path"].iloc[0]
        pixels = pydicom.dcmread(path).pixel_array[::STEP, ::STEP].astype(np.float32)
        low, high = np.percentile(pixels[pixels > 0], [1, 99.5])
        ax.imshow(np.clip((pixels - low) / (high - low), 0, 1), cmap="gray")
        for points in masses.loc[masses.file_id == file_id, "points"]:
            closed = np.vstack([points, points[:1]]) / STEP
            ax.plot(closed[:, 0], closed[:, 1], "r-", lw=1.2)
        ax.set_title(file_id, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_masses(list(RNG.choice(masses.file_id.unique(), size=4, replace=False)))

## 7. Checks to carry into the data pipeline

- Polygons that leave the image by a few pixels (clip when rasterizing).
- Mixed spelling of ROI names (`Spiculated Region`, `Espiculated Region`, ...).
- Right-breast images have the chest wall on the right, left-breast images on
  the left; flip one side to a common orientation before training.
